# Duplicate Detection

This notebook identifies duplicate records across the raw Instacart datasets.

The analysis distinguishes between true duplicate records and legitimate repeated keys that occur naturally in transactional tables.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
DATA_DIR = Path("../data/raw")

DATA_FILES = {
    "orders": "orders.csv",
    "order_products_prior": "order_products__prior.csv",
    "order_products_train": "order_products__train.csv",
    "products": "products.csv",
    "aisles": "aisles.csv",
    "departments": "departments.csv",
}

## Exact Duplicate Records

Exact duplicate rows are checked across each raw dataset.

Large transaction tables are processed in chunks to avoid loading the complete dataset into memory.

In [3]:
def count_exact_duplicates(path, chunksize=200_000):
    duplicate_count = 0

    for chunk in pd.read_csv(path, chunksize=chunksize):
        duplicate_count += chunk.duplicated().sum()

    return int(duplicate_count)


exact_duplicates = {}

for name, filename in DATA_FILES.items():
    exact_duplicates[name] = count_exact_duplicates(
        DATA_DIR / filename
    )

exact_duplicates

{'orders': 0,
 'order_products_prior': 0,
 'order_products_train': 0,
 'products': 0,
 'aisles': 0,
 'departments': 0}

## Key-Level Duplicate Analysis

Repeated keys are not automatically treated as duplicates.

For example, an `order_id` can legitimately appear multiple times in an order-products table because a single order may contain multiple products.

Therefore, key-level duplication is evaluated according to the expected role of each dataset.

In [4]:
orders = pd.read_csv(
    DATA_DIR / DATA_FILES["orders"],
    usecols=["order_id", "user_id"]
)

products = pd.read_csv(
    DATA_DIR / DATA_FILES["products"],
    usecols=["product_id"]
)

aisles = pd.read_csv(
    DATA_DIR / DATA_FILES["aisles"],
    usecols=["aisle_id"]
)

departments = pd.read_csv(
    DATA_DIR / DATA_FILES["departments"],
    usecols=["department_id"]
)

key_duplicates = {
    "orders.order_id": int(orders["order_id"].duplicated().sum()),
    "products.product_id": int(products["product_id"].duplicated().sum()),
    "aisles.aisle_id": int(aisles["aisle_id"].duplicated().sum()),
    "departments.department_id": int(
        departments["department_id"].duplicated().sum()
    ),
}

key_duplicates

{'orders.order_id': 0,
 'products.product_id': 0,
 'aisles.aisle_id': 0,
 'departments.department_id': 0}

In [5]:
def count_duplicate_order_products(path, chunksize=200_000):
    duplicate_count = 0

    for chunk in pd.read_csv(
        path,
        usecols=["order_id", "product_id"],
        chunksize=chunksize
    ):
        duplicate_count += chunk.duplicated(
            subset=["order_id", "product_id"]
        ).sum()

    return int(duplicate_count)


transaction_duplicates = {
    "order_products_prior": count_duplicate_order_products(
        DATA_DIR / DATA_FILES["order_products_prior"]
    ),
    "order_products_train": count_duplicate_order_products(
        DATA_DIR / DATA_FILES["order_products_train"]
    ),
}

transaction_duplicates

{'order_products_prior': 0, 'order_products_train': 0}

In [6]:
duplicate_summary = pd.DataFrame({
    "Check": [
        "Orders exact duplicate rows",
        "Orders duplicate order_id",
        "Products duplicate product_id",
        "Aisles duplicate aisle_id",
        "Departments duplicate department_id",
        "Prior duplicate order-product pairs",
        "Train duplicate order-product pairs",
    ],
    "Count": [
        exact_duplicates["orders"],
        key_duplicates["orders.order_id"],
        key_duplicates["products.product_id"],
        key_duplicates["aisles.aisle_id"],
        key_duplicates["departments.department_id"],
        transaction_duplicates["order_products_prior"],
        transaction_duplicates["order_products_train"],
    ]
})

duplicate_summary

,Check,Count
0,Orders exact duplicate rows,0
1,Orders duplicate order_id,0
2,Products duplicate product_id,0
3,Aisles duplicate aisle_id,0
4,Departments duplicate department_id,0
5,Prior duplicate order-product pairs,0
6,Train duplicate order-product pairs,0
